# Night Vapor Dynamics (CasADi)

Fast dynamics implementation using CasADi for automatic differentiation and code generation.

In [ ]:
import casadi as ca
import numpy as np

# Define smooth approximation functions in CasADi
def smooth_max_casadi(a, b, epsilon=1e-3):
    """Smooth approximation of max(a,b): (a + b + sqrt((a-b)² + ε²))/2"""
    return (a + b + ca.sqrt((a - b)**2 + epsilon**2)) / 2

def smooth_min_casadi(a, b, epsilon=1e-3):
    """Smooth approximation of min(a,b): (a + b - sqrt((a-b)² + ε²))/2"""
    return (a + b - ca.sqrt((a - b)**2 + epsilon**2)) / 2

def smooth_clamp_casadi(x, min_val, max_val, epsilon=1e-3):
    """Smooth approximation of clamping x to [min_val, max_val]"""
    return smooth_min_casadi(smooth_max_casadi(x, min_val, epsilon), max_val, epsilon)

def smooth_abs_casadi(x, epsilon=1e-6):
    """Smooth approximation of absolute value: sqrt(x² + ε²)"""
    return ca.sqrt(x**2 + epsilon**2)

def smooth_sign_casadi(x, epsilon=1e-3):
    """Smooth approximation of sign function: tanh(x/ε)"""
    return ca.tanh(x / epsilon)

print("Creating CasADi dynamics with smooth approximations...")

In [ ]:
# Parameter values
params = {
    'thr_max': 0.32,
    'm': 0.025,
    'S': 0.025,
    'rho': 1.225,
    'g': 9.81,
    'Jx': 1.0e-4,
    'Jy': 1.0e-4,
    'Jz': 1.0e-4,
    'cbar': 0.09,
    'span': 0.34,
    'Cm0': 0.01,
    'Cldr': 0.15,
    'Cmde': 0.25,
    'Cndr': 0.10,
    'CYdr': -0.08,
    'CL0': 0.6,
    'CLa': 4.8,
    'Cma': -0.12,
    'Cmq': -0.1,
    'CD0': 0.10,
    'CDCLS': 0.12,
    'Cnb': 0.150,
    'Clp': -0.11,
    'Cnr': -0.105,
    'Cnp': -0.15,
    'Clr': 0.10,
    'CYb': -0.02,
    'CYr': 0.2,
    'CYp': 0.1,
}

# Control surface limits
DEG2RAD = np.pi / 180
max_defl = 30 * DEG2RAD
max_defl_elev = 24 * DEG2RAD

In [ ]:
# Define symbolic variables
# State: [px, py, pz, u, v, w, qw, qx, qy, qz, p, q, r]
x = ca.MX.sym('x', 13)
# Control: [throttle, elevator, rudder]
u = ca.MX.sym('u', 3)

# Extract state components
px, py, pz = x[0], x[1], x[2]
u_vel, v_vel, w_vel = x[3], x[4], x[5]  # Renamed to avoid conflict with control
qw, qx, qy, qz = x[6], x[7], x[8], x[9]
p, q, r = x[10], x[11], x[12]

# Extract control components
throttle, elevator, rudder = u[0], u[1], u[2]

print("Symbolic variables defined")

In [ ]:
# Control processing with SMOOTH saturation
# Throttle saturation (minimum 1e-3) - smooth version
throttle_sat = smooth_max_casadi(throttle, 1e-3, epsilon=1e-6)

# Control surface deflections
elev_rad = max_defl_elev * elevator
rud_rad = max_defl * rudder

# Velocity saturation (SMOOTH)
vel_limit = 5.0
u_sat = smooth_clamp_casadi(u_vel, -vel_limit, vel_limit, epsilon=1e-3)
v_sat = smooth_clamp_casadi(v_vel, -vel_limit, vel_limit, epsilon=1e-3)
w_sat = smooth_clamp_casadi(w_vel, -vel_limit, vel_limit, epsilon=1e-3)

# Airspeed calculation with tolerance (SMOOTH)
tol_v = 0.1
V = ca.sqrt(u_sat**2 + v_sat**2 + w_sat**2)
V_safe = smooth_max_casadi(V, tol_v, epsilon=1e-3)
u_safe = smooth_max_casadi(smooth_abs_casadi(u_sat), tol_v) * smooth_sign_casadi(u_sat)

# Angle of attack and sideslip
alpha = ca.atan2(-w_sat, u_safe)
beta = ca.asin(v_sat / V_safe)

# Angle saturation (SMOOTH)
alpha_max = 45 * DEG2RAD
alpha_min = -30 * DEG2RAD
beta_max = 30 * DEG2RAD

alpha = smooth_clamp_casadi(alpha, alpha_min, alpha_max, epsilon=1e-3)
beta = smooth_clamp_casadi(beta, -beta_max, beta_max, epsilon=1e-3)

# Dynamic pressure
qbar = 0.5 * params['rho'] * V_safe**2

print("Control processing and airspeed calculation done (SMOOTH)")

In [ ]:
# Aerodynamic coefficients
CL = params['CL0'] + params['CLa'] * alpha
CD = params['CD0'] + params['CDCLS'] * CL**2
CY = -(params['CYb'] * beta) + (params['CYdr'] * rud_rad / max_defl) + \
     ((params['span'] / (2 * V_safe)) * ((params['CYp'] * p) + (params['CYr'] * r)))

# Aerodynamic forces in wind frame
Dw = qbar * params['S'] * CD  # Drag
Lw = qbar * params['S'] * CL  # Lift
Yw = qbar * params['S'] * CY  # Side force

print("Aerodynamic coefficients calculated")

In [ ]:
# Rotation matrices using CasADi
def rotation_matrix_casadi(qw, qx, qy, qz):
    """Rotation matrix from quaternion"""
    R11 = 1 - 2*(qy**2 + qz**2)
    R12 = 2*(qx*qy - qw*qz)
    R13 = 2*(qx*qz + qw*qy)
    R21 = 2*(qx*qy + qw*qz)
    R22 = 1 - 2*(qx**2 + qz**2)
    R23 = 2*(qy*qz - qw*qx)
    R31 = 2*(qx*qz - qw*qy)
    R32 = 2*(qy*qz + qw*qx)
    R33 = 1 - 2*(qx**2 + qy**2)
    return ca.vertcat(
        ca.horzcat(R11, R12, R13),
        ca.horzcat(R21, R22, R23),
        ca.horzcat(R31, R32, R33)
    )

# Wind to body rotation
cos_half_alpha = ca.cos(alpha/2)
sin_half_alpha = ca.sin(alpha/2)
cos_half_beta = ca.cos(beta/2)
sin_half_beta = ca.sin(beta/2)

qw_nb = cos_half_beta * cos_half_alpha
qx_nb = -cos_half_beta * sin_half_alpha
qy_nb = -sin_half_beta * cos_half_alpha
qz_nb = -(-sin_half_beta * sin_half_alpha)

R_nb = rotation_matrix_casadi(qw_nb, qx_nb, qy_nb, qz_nb)

# Transform forces from wind to body frame
F_wind = ca.vertcat(-Dw, Yw, Lw)
F_aero_b = R_nb @ F_wind

print("Rotation matrices and force transformation done")

In [ ]:
# Thrust and gravity forces
T_b = ca.vertcat(params['thr_max'] * throttle_sat, 0, 0)

# Gravity in body frame
R_wb = rotation_matrix_casadi(qw, qx, qy, qz)
R_bw = R_wb.T
W_b = R_bw @ ca.vertcat(0, 0, -params['m'] * params['g'])

# Total force
F_total = F_aero_b + T_b + W_b

print("Forces calculated")

In [ ]:
# Moments
Cl = (-1) * params['Cldr'] * rud_rad
Cm = params['Cm0'] + (params['Cma'] * alpha) + (params['Cmde'] * elev_rad)
Cn = (params['Cnb'] * beta) + (params['Cndr'] * rud_rad)

# Basic aerodynamic moments
Mx_aero = qbar * params['S'] * params['span'] * Cl
My_aero = qbar * params['S'] * params['cbar'] * Cm
Mz_aero = qbar * params['S'] * params['span'] * Cn

# Damping moments
Mx_damp = (params['Clp'] * (params['span'] / (2 * V_safe)) * p) + \
          (params['Clr'] * (params['span'] / (2 * V_safe)) * r)
My_damp = (params['Cmq'] * (params['cbar'] / (2 * V_safe)) * q)
Mz_damp = (params['Cnp'] * (params['span'] / (2 * V_safe)) * p) + \
          (params['Cnr'] * (params['span'] / (2 * V_safe)) * r)

# Total moments
M_total = ca.vertcat(
    Mx_aero + Mx_damp,
    My_aero + My_damp,
    Mz_aero + Mz_damp
)

print("Moments calculated")

In [ ]:
# Dynamics equations
# Position kinematics
v_b = ca.vertcat(u_sat, v_sat, w_sat)
p_dot = R_wb @ v_b

# Translational dynamics
omega_b = ca.vertcat(p, q, r)
omega_skew = ca.vertcat(
    ca.horzcat(0, -r, q),
    ca.horzcat(r, 0, -p),
    ca.horzcat(-q, p, 0)
)
v_b_dot = (1/params['m']) * F_total - omega_skew @ v_b

# Quaternion kinematics
q_norm = ca.sqrt(qw**2 + qx**2 + qy**2 + qz**2)
qw_n, qx_n, qy_n, qz_n = qw/q_norm, qx/q_norm, qy/q_norm, qz/q_norm

Omega = ca.vertcat(
    ca.horzcat(0, -p, -q, -r),
    ca.horzcat(p, 0, r, -q),
    ca.horzcat(q, -r, 0, p),
    ca.horzcat(r, q, -p, 0)
)
q_vec_dot = 0.5 * Omega @ ca.vertcat(qw_n, qx_n, qy_n, qz_n)

# Rotational dynamics
J = ca.diag(ca.vertcat(params['Jx'], params['Jy'], params['Jz']))
J_omega = J @ omega_b
omega_b_dot = ca.solve(J, M_total - omega_skew @ J_omega)

# Complete dynamics vector
f_casadi = ca.vertcat(
    p_dot,          # px_dot, py_dot, pz_dot
    v_b_dot,        # u_dot, v_dot, w_dot
    q_vec_dot,      # qw_dot, qx_dot, qy_dot, qz_dot
    omega_b_dot     # p_dot, q_dot, r_dot
)

print("Dynamics equations assembled")

In [ ]:
# Create CasADi functions (FAST!)
print("Creating CasADi functions...")

# Dynamics function
f_func = ca.Function('f', [x, u], [f_casadi])

# Jacobian function (automatic differentiation - very fast!)
F_jacobian = ca.jacobian(f_casadi, x)
F_func = ca.Function('F', [x, u], [F_jacobian])

print("✅ CasADi functions created successfully!")
print(f"✅ Dynamics function: f_func(x, u) -> (13,1)")
print(f"✅ Jacobian function: F_func(x, u) -> (13,13)")

In [ ]:
# Test the functions
print("Testing CasADi functions...")

# Test values
x_test = np.array([0, 0, -100, 15, 0, 1, 1, 0, 0, 0, 0, 0, 0])  # Sample state
u_test = np.array([0.5, 0.1, 0.05])  # Sample control

try:
    # Test dynamics
    f_val = f_func(x_test, u_test)
    print(f"✅ Dynamics evaluation successful: shape {f_val.shape}")
    
    # Test Jacobian
    F_val = F_func(x_test, u_test)
    print(f"✅ Jacobian evaluation successful: shape {F_val.shape}")
    
    print(f"Sample dynamics values: {np.array(f_val[:5]).flatten()}")
    
except Exception as e:
    print(f"✗ Error: {e}")

# Define state and control dimensions for compatibility
X = ca.MX.sym('X_dummy', 13)  # For len(X) compatibility
U = ca.MX.sym('U_dummy', 3)   # For len(U) compatibility

print("🚀 CasADi dynamics ready for state estimation!")